In [1]:
import pandas as pd 
import numpy as np 
import scanpy as sc 
import os 
import warnings 
warnings.filterwarnings("ignore") 
import matplotlib as mpl 
import matplotlib.pyplot as plt 
import PROST 
PROST.__version__

' 1.1.2 '

In [2]:
# the location of R (used for the mclust clustering)
ENVpath = "/home/ajita/anaconda3/envs/PROST_ENV"      # refer to 'How to use PROST' section

os.environ['R_HOME'] = '/usr/lib/R'
os.environ['R_USER'] = f'{ENVpath}/lib/python3.7/site-packages/rpy2'

# Set seed
SEED = 818
PROST.setup_seed(SEED)


n_clusters = 11


In [3]:
# We are only saving clusters labels...

def save_files(adata, results_path):
    
    #adata.write_h5ad(output_dir+"/PNN_result.h5")   
    #clustering = adata.obs["clustering"]
    #clustering.to_csv(output_dir+"/clusters.csv",header = False)
    
    pp_clustering = adata.obs["pp_clustering"] 
    pp_clustering.to_csv(results_path, header = True)
    
    #embedding = adata.obsm["PROST"]
    #np.savetxt(output_dir+"/embedding.txt",embedding) 
    #adata = sc.read(output_dir+'/PNN_result.h5')



In [4]:

method_name = "PROST"

path = "/data/Ajita/Spatial/Datasets/Spatial_Domain/Real_datasets/mouse_breast_cancer/mouse_breast_cancer.h5ad"

results_path = "/data/Ajita/Spatial/Datasets/Spatial_Domain/Real_datasets/mouse_breast_cancer/final_domain_outputs/" \
+ "/output_" + method_name + ".csv"
adata = sc.read_h5ad(path)
    
adata.var_names_make_unique()


adata = PROST.prepare_for_PI(adata, percentage = 0.01, platform="Merfish") 
adata = PROST.cal_PI(adata, kernel_size=6, platform="Merfish")

#adata.write_h5ad("./Outputs/" + data_name + "_" + str(id) +"/PI_result.h5")
#adata = sc.read_h5ad("./Outputs/" + data_name + "_" + str(id) +"/PI_result.h5")

sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata = PROST.feature_selection(adata, by = "prost", n_top_genes = 3000)


PROST.run_PNN(adata,
            key_added = "PROST",
            init="mclust",                         
            n_clusters = n_clusters,  
            adj_mode = "neighbour",
            k_neighbors = 7, 
            lap_filter = 2,                                  
            lr = 0.1,                         
            SEED=SEED,                          
            max_epochs = 500,                        
            tol = 5e-3,                        
            post_processing = True,                        
            pp_run_times = 3,
            cuda = False)

save_files(adata, results_path)



Filtering genes ...
Spatial gene expression is interpolated into images of size [450 * 532]

Normalize each geneing...

Gaussian filtering...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10575/10575 [06:24<00:00, 27.49it/s]



Binary segmentation for each gene:


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10575/10575 [02:32<00:00, 69.18it/s]



Spliting subregions for each gene:


...


Computing PROST Index for each gene:


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10575/10575 [12:52<00:00, 13.69it/s]



PROST Index calculation completed !!

Calculating adjacency matrix, mode=neighbour, k_neighbors=7...

Running PCA ...
Laplacian Smoothing ...

Initializing cluster centers with mclust, n_clusters known
fitting ...

  |                                                                            
  |                                                                      |   0%
  |                                                                            
  |===================================                                   |  50%
  |                                                                            
  |======================================================================| 100%




Epoch: : 501it [00:13, 37.96it/s, loss=0.18447672]                                                                                                                           


Clustering completed !!

Post-processing for clustering result ...
Refining clusters, run times: 1/3
Refining clusters, run times: 2/3
Refining clusters, run times: 3/3
